# VROOM-SBI: Notebook Containing All the Paper Figures

**VROOM-SBI: A Fast Simulation-Based Bayesian Inference Methodology for QU-Fitting**

This notebook reproduces every figure in the paper, in the order they appear.
Each section explains the scientific motivation and the code that generates the plot.

---

## Data tiers

| Tier | What you need | Figures |
|------|--------------|--------|
| **Tier 1** | Just this repo (`models/*.pt`) | SBC, recovery, N=2, spectral demo |
| **Tier 2** | Cached qufit posteriors (`scripts/qufit_cache_*.npz`) | VROOM-SBI vs RMtools corners |
| **Tier 3** | Full G71 FITS cubes (set `$G71_DIR`) | RM maps, spectral index map |

Tier 3 cells are clearly labelled. They will print a warning and skip gracefully if the data is absent.

Set the `G71_DIR` environment variable to the directory containing the G71 FITS cubes
before launching Jupyter, or edit `G71_DIR` directly in the setup cell below.

---

## How to run

```bash
# From the repo root — interactive
pixi run -e notebooks jupyter notebook notebooks/paper_figures.ipynb
```

Or run non-interactively:
```bash
pixi run -e notebooks jupyter nbconvert --to notebook --execute notebooks/paper_figures.ipynb
```


In [ ]:
# ── Environment setup ─────────────────────────────────────────────────────────
import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import torch
from scipy.stats import qmc, gaussian_kde

REPO = Path("..").resolve()   # notebook lives in notebooks/; repo root is one level up
sys.path.insert(0, str(REPO))

MODELS_DIR = REPO / "models"
IMAGES_DIR = REPO / "images"
IMAGES_DIR.mkdir(exist_ok=True)

# ── Tier 3 data directory (MACS J1752+4440 / G71) ────────────────────────────
# Set the G71_DIR environment variable before launching Jupyter, or edit this line.
G71_DIR = Path(os.environ.get("G71_DIR", ""))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
print(f"Repo   : {REPO}")
print(f"G71_DIR: {G71_DIR}  (exists: {G71_DIR.exists() if str(G71_DIR) else 'not set'})")
print("Models :")
for p in sorted(MODELS_DIR.glob("*.pt")):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.1f} MB)")

# ── Colour palette (consistent across all figures) ────────────────────────────
BLUE   = "#3a86ff"
ORANGE = "#e07a5f"
RED    = "#e63946"
GREEN  = "#2ca02c"
GREY   = "#888888"
SBI_COL   = "#2166ac"
RM_COL    = "#d6604d"
TRUTH_COL = "#1a9641"

# ── Global matplotlib style ───────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "serif",
    "font.size":          9,
    "axes.labelsize":     9,
    "axes.titlesize":     9,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "axes.linewidth":     0.6,
    "xtick.major.width":  0.6,
    "ytick.major.width":  0.6,
    "lines.linewidth":    1.0,
    "pdf.fonttype":       42,
    "ps.fonttype":        42,
})


In [ ]:
# ── Shared utility functions used by multiple figure cells ────────────────────

from src.inference.engine import load_posterior
from src.simulator.base_simulator import RMSimulator


def sobol_prior(lo, hi, n):
    """Draw n quasi-random samples from Uniform(lo, hi) using a Sobol sequence."""
    lo, hi = np.array(lo), np.array(hi)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        u = qmc.Sobol(d=len(lo), scramble=True).random(n)
    return (lo + u * (hi - lo)).astype(np.float32)


def make_freq_file(lambda_sq, tmp_path="/tmp/vroom_nb_freqs.txt"):
    """Convert lambda^2 array (m^2) to frequency (Hz) and write to a temp file."""
    freqs = 3e8 / np.sqrt(np.array(lambda_sq))
    np.savetxt(tmp_path, freqs)
    return tmp_path


def coverage_at_68(samples, truths):
    """Empirical fraction of test cases where truth falls inside the 16th-84th pct interval."""
    lo = np.percentile(samples, 16, axis=1)
    hi = np.percentile(samples, 84, axis=1)
    return float(np.mean((truths >= lo) & (truths <= hi)))


def _get_range(vals, pad=0.12):
    lo, hi = vals.min(), vals.max()
    d = (hi - lo) * pad
    return lo - d, hi + d


def _draw_kde_contours(ax, xv, yv, xr, yr, color=BLUE, levels=(0.68, 0.95)):
    """Draw filled + outlined KDE contours at the requested probability levels."""
    try:
        kde = gaussian_kde([xv, yv], bw_method="scott")
        xg = np.linspace(*xr, 150)
        yg = np.linspace(*yr, 150)
        Xg, Yg = np.meshgrid(xg, yg)
        Z  = kde(np.vstack([Xg.ravel(), Yg.ravel()])).reshape(Xg.shape)
        Zf = np.sort(Z.ravel())[::-1]
        cum = np.cumsum(Zf) / Zf.sum()
        lvls = sorted({float(Zf[np.searchsorted(cum, lv)]) for lv in levels})
        if lvls:
            ax.contourf(Xg, Yg, Z, levels=[lvls[0], Z.max()], colors=[color], alpha=0.18)
        ax.contour(Xg, Yg, Z, levels=lvls, colors=[color], linewidths=0.9)
    except Exception:
        pass


def draw_corner(axes_grid, samples, truths, n_params, param_labels, color=BLUE):
    """Lower-triangle corner plot of `samples` (N, n_params) with truth lines."""
    ranges = [_get_range(samples[:, i]) for i in range(n_params)]
    for row in range(n_params):
        for col in range(n_params):
            ax = axes_grid[row, col]
            if col > row:
                ax.set_visible(False)
                continue
            if row == col:
                vals = samples[:, row]
                ax.hist(vals, bins=35, range=ranges[row],
                        color=color, alpha=0.45, density=True, linewidth=0)
                try:
                    kde = gaussian_kde(vals, bw_method=0.3)
                    xp  = np.linspace(*ranges[row], 250)
                    ax.plot(xp, kde(xp), color=color, lw=1.2)
                except Exception:
                    pass
                ax.axvline(truths[row], color=RED, lw=1.2, ls="--")
                ax.set_xlim(ranges[row])
                ax.set_yticks([])
            else:
                _draw_kde_contours(ax, samples[:, col], samples[:, row],
                                   ranges[col], ranges[row], color=color)
                ax.axvline(truths[col], color=RED, lw=0.8, ls="--", alpha=0.7)
                ax.axhline(truths[row], color=RED, lw=0.8, ls="--", alpha=0.7)
                ax.set_xlim(ranges[col])
                ax.set_ylim(ranges[row])
            if row == n_params - 1:
                ax.set_xlabel(param_labels[col], labelpad=3)
            else:
                ax.set_xticklabels([])
            if col == 0 and row != 0:
                ax.set_ylabel(param_labels[row], labelpad=3)
            else:
                ax.set_yticklabels([])
            ax.tick_params(direction="in", top=True, right=True)


def compute_weights(noise_per_chan):
    """Inverse-variance weights w_k = min(1, (sigma_med / sigma_k)^2)."""
    good = noise_per_chan > 0
    w = np.zeros_like(noise_per_chan, dtype=np.float32)
    if good.any():
        s_med = np.median(noise_per_chan[good])
        w[good] = np.minimum(1.0, (s_med / noise_per_chan[good]) ** 2)
    return w


print("Helpers loaded.")


---
## 1. Simulation-Based Calibration (SBC)

**What is SBC?**  
SBC (Talts et al. 2018) is a self-consistency check for any Bayesian inference algorithm. The idea:

1. Draw a true parameter `theta` from the prior.
2. Simulate an observation `x` from the likelihood `p(x | theta)`.
3. Draw `N_POST` posterior samples `{theta_i}` given `x`.
4. Compute the *rank* of the true `theta` among the posterior samples.

If the posterior is **calibrated**, the rank is uniformly distributed on `[0, N_POST]`.  
- **Flat histogram** → well-calibrated posterior.
- **U-shaped histogram** → over-confident (posterior too narrow).
- **Hump in the middle** → under-confident (posterior too wide).

We do this for all four single-component depolarisation models:
| Model | Parameters |
|-------|------------|
| `faraday_thin` | φ, p₀, χ₀ |
| `internal_dispersion` | φ, σ_φ, p₀, χ₀ |
| `external_dispersion` | φ, σ_φ, p₀, χ₀ |
| `burn_slab` | φ_c, Δφ, p₀, χ₀ |

**Figure**: `images/sbc_ranks.pdf`


In [ ]:
# ── SBC rank histograms for all four N=1 depolarisation models ────────────────
# Reproduces: images/sbc_ranks.pdf (scripts/plot_sbc.py)

N_SBC  = 500    # number of (theta, x) test draws
N_POST = 1000   # posterior samples per observation
SEED   = 7

SIGMA_MIN, SIGMA_MAX = 0.0005, 0.1   # additive noise range (fractional pol. units)

np.random.seed(SEED)
torch.manual_seed(SEED)

PARAM_LABELS_SBC = {
    "faraday_thin":        [r"$\phi$",   r"$p_0$",             r"$\chi_0$"],
    "internal_dispersion": [r"$\phi$",   r"$\sigma_\phi$",     r"$p_0$",  r"$\chi_0$"],
    "external_dispersion": [r"$\phi$",   r"$\sigma_\phi$",     r"$p_0$",  r"$\chi_0$"],
    "burn_slab":           [r"$\phi_c$", r"$\Delta\phi$",      r"$p_0$",  r"$\chi_0$"],
}
MODEL_TITLES_SBC = {
    "faraday_thin":        "Faraday-thin",
    "internal_dispersion": "Internal dispersion",
    "external_dispersion": "External dispersion",
    "burn_slab":           "Burn slab",
}
MODEL_ORDER = ["faraday_thin", "internal_dispersion", "external_dispersion", "burn_slab"]

model_ranks  = {}
model_params = {}

for model_type in MODEL_ORDER:
    pt = MODELS_DIR / f"posterior_{model_type}_n1.pt"
    if not pt.exists():
        print(f"[skip] {pt.name} not found")
        continue
    print(f"\n=== {model_type} ===")

    posterior, meta = load_posterior(pt, device=DEVICE)
    input_channels  = int(meta.get("input_channels", 2))
    n_freq          = int(meta.get("n_freq", 127))
    lambda_sq       = np.array(meta["lambda_sq"])
    pb              = meta["prior_bounds"]
    lo, hi          = np.array(pb["low"]), np.array(pb["high"])

    freq_file = make_freq_file(lambda_sq)
    thetas    = sobol_prior(lo, hi, N_SBC)                     # (N_SBC, n_params)
    sim       = RMSimulator(freq_file, n_components=1, model_type=model_type)
    weights   = np.ones(n_freq, dtype=np.float32)
    rng       = np.random.default_rng(SEED + 1)
    sigma_bases = rng.uniform(SIGMA_MIN, SIGMA_MAX, N_SBC).astype(np.float32)

    n_params = thetas.shape[1]
    ranks    = np.zeros((N_SBC, n_params), dtype=int)
    for i in range(N_SBC):
        qu = sim.simulate_batch(thetas[i:i+1], weights[None], noise_sigma=sigma_bases[i])[0]
        x  = np.concatenate([qu, weights]) if input_channels == 3 else qu
        x_t = torch.tensor(x, dtype=torch.float32, device=DEVICE)
        samps = posterior.sample((N_POST,), x=x_t).cpu().numpy()
        for p in range(n_params):
            ranks[i, p] = int(np.sum(samps[:, p] < thetas[i, p]))
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{N_SBC}")

    model_ranks[model_type]  = ranks
    model_params[model_type] = PARAM_LABELS_SBC[model_type]

# ── Figure layout: one row per model, one column per parameter ─────────────────
N_BINS   = 20
expected = N_SBC / N_BINS
lo_band  = expected * 0.5
hi_band  = expected * 1.5

max_params = max(len(v) for v in model_params.values())
n_rows     = len(model_ranks)
n_cols     = max_params
fig_w, fig_h = 2.5 * n_cols, 2.2 * n_rows

fig      = plt.figure(figsize=(fig_w, fig_h))
outer_gs = fig.add_gridspec(
    n_rows, n_cols, hspace=0.55, wspace=0.25,
    top=0.93, bottom=0.07, left=0.07, right=0.98,
)

def draw_bar(ax, counts, edges, ymax):
    ax.bar(edges[:-1], counts, width=edges[1]-edges[0],
           color=BLUE, alpha=0.65, linewidth=0, align="edge")
    ax.axhline(expected, color=ORANGE, lw=1.2, ls="--")
    ax.axhspan(lo_band, hi_band, color=ORANGE, alpha=0.12)
    ax.set_xlim(0, N_POST)
    ax.set_ylim(0, ymax)

row_idx = 0
for model_type in MODEL_ORDER:
    if model_type not in model_ranks:
        continue
    labels   = model_params[model_type]
    n_params = len(labels)
    for col in range(n_cols):
        cell = outer_gs[row_idx, col]
        if col >= n_params:
            fig.add_subplot(cell).set_visible(False)
            continue
        counts, edges = np.histogram(model_ranks[model_type][:, col],
                                     bins=N_BINS, range=(0, N_POST))
        ax = fig.add_subplot(cell)
        ymax = max(counts.max() * 1.15, expected * 2.8)
        draw_bar(ax, counts, edges, ymax)
        ax.set_xlabel(labels[col], labelpad=2)
        if col == 0:
            ax.set_ylabel(MODEL_TITLES_SBC[model_type], labelpad=4, fontsize=8)
            ax.set_yticks([0, int(expected), int(expected * 2)])
        else:
            ax.set_yticks([])
        if row_idx == 0:
            ax.set_title(labels[col], pad=3, fontsize=7)
        ax.tick_params(direction="in", top=True, right=True)
    row_idx += 1

fig.suptitle("SBC rank histograms — flat = calibrated posterior", fontsize=9, y=0.99)
fig.savefig(IMAGES_DIR / "sbc_ranks.pdf", bbox_inches="tight", dpi=200)
plt.show()
print(f"Saved → {IMAGES_DIR / 'sbc_ranks.pdf'}")


---
## 2. Single-Component Parameter Recovery

Recovery plots show how well the **posterior median** recovers the truth on a held-out test set.

For each model we:
1. Draw `N_TEST = 200` parameter vectors `theta` from the prior using a Sobol sequence.
2. Simulate a noisy Q/U spectrum with a random noise level `sigma ~ Uniform(0.0005, 0.1)`.
3. Draw `N_POST = 1000` posterior samples for each simulated spectrum.
4. Report, per parameter:
   - **MedAE**: median absolute error (smaller = more accurate).
   - **Bias**: mean signed error (near 0 = unbiased).
   - **Cov**: empirical coverage of the 68% credible interval (should be ≈ 68%).

**Top row**: recovered (posterior median) vs. true value. Points should lie on the dashed diagonal.  
**Bottom row**: residuals (recovered − true) vs. true. Orange band = ±MedAE.

**Figures**: `images/posterior_{model_type}_n1_recovery.pdf` for each of the four models.


In [ ]:
# ── Shared recovery infrastructure ────────────────────────────────────────────

N_TEST    = 200
N_POST_R  = 1000
SEED_R    = 0
SIGMA_MIN_R = 0.001 * 0.5
SIGMA_MAX_R = 0.05  * 2.0

PARAM_LABELS_R = {
    "faraday_thin":        [r"$\phi$",   r"$p_0$",           r"$\chi_0$"],
    "internal_dispersion": [r"$\phi$",   r"$\sigma_\phi$",   r"$p_0$",  r"$\chi_0$"],
    "external_dispersion": [r"$\phi$",   r"$\sigma_\phi$",   r"$p_0$",  r"$\chi_0$"],
    "burn_slab":           [r"$\phi_c$", r"$\Delta\phi$",    r"$p_0$",  r"$\chi_0$"],
}
PARAM_UNITS_R = {
    "faraday_thin":        [r"rad m$^{-2}$", "",               "rad"],
    "internal_dispersion": [r"rad m$^{-2}$", r"rad m$^{-2}$", "",   "rad"],
    "external_dispersion": [r"rad m$^{-2}$", r"rad m$^{-2}$", "",   "rad"],
    "burn_slab":           [r"rad m$^{-2}$", r"rad m$^{-2}$", "",   "rad"],
}
MODEL_TITLES_R = {
    "faraday_thin":        "Faraday-thin",
    "internal_dispersion": "Internal Faraday dispersion",
    "external_dispersion": "External Faraday dispersion",
    "burn_slab":           "Burn slab",
}


def make_recovery_figure(model_type, thetas, all_samples, lo, hi):
    """
    Two-row recovery figure for a single model.
    Row 0: recovered (posterior median) vs true.
    Row 1: residuals vs true.
    Each column = one parameter.
    """
    n_params = thetas.shape[1]
    labels   = PARAM_LABELS_R[model_type]
    units    = PARAM_UNITS_R[model_type]
    lo, hi   = np.array(lo), np.array(hi)

    med    = np.median(all_samples, axis=1)         # (n_test, n_params)
    p16    = np.percentile(all_samples, 16, axis=1)
    p84    = np.percentile(all_samples, 84, axis=1)
    err_lo = med - p16
    err_hi = p84 - med
    resid  = med - thetas

    fig_w = n_params * 2.2
    fig   = plt.figure(figsize=(fig_w, 5.2))
    fig.suptitle(MODEL_TITLES_R[model_type], fontsize=10, fontweight="bold", y=0.98)

    gs = gridspec.GridSpec(
        2, n_params, figure=fig, hspace=0.10, wspace=0.38,
        top=0.91, bottom=0.12, left=0.10, right=0.98,
    )
    gs.set_height_ratios([2.6, 1.0])

    pt_kw = dict(fmt="o", ms=1.8, alpha=0.55, color=ORANGE,
                 ecolor=ORANGE, elinewidth=0.5, capsize=0, capthick=0)

    for pi in range(n_params):
        ax_top = fig.add_subplot(gs[0, pi])
        ax_bot = fig.add_subplot(gs[1, pi], sharex=ax_top)

        true_v = thetas[:, pi]
        med_v  = med[:, pi]
        res_v  = resid[:, pi]
        elo    = err_lo[:, pi]
        ehi    = err_hi[:, pi]

        medae = float(np.median(np.abs(res_v)))
        bias  = float(np.mean(res_v))
        cov68 = coverage_at_68(all_samples[:, :, pi], true_v)

        unit_str = units[pi]
        xlabel   = labels[pi] + (f" ({unit_str})" if unit_str else "")
        pad = 0.04 * (hi[pi] - lo[pi])
        lim = [lo[pi] - pad, hi[pi] + pad]

        # top: recovered vs true
        ax_top.errorbar(true_v, med_v, yerr=[elo, ehi], **pt_kw)
        ax_top.plot(lim, lim, "k--", lw=0.8, alpha=0.7)
        ax_top.set_xlim(lim); ax_top.set_ylim(lim)
        ax_top.set_ylabel("Recovered" if pi == 0 else "", labelpad=3)
        ax_top.tick_params(labelbottom=False, direction="in", top=True, right=True)
        ax_top.set_title(labels[pi], pad=3)
        ax_top.grid(True, lw=0.4, alpha=0.4)
        txt = f"MedAE={medae:.2g}\nBias={bias:+.2g}\nCov={cov68:.0%}"
        ax_top.text(0.97, 0.04, txt, transform=ax_top.transAxes,
                    fontsize=6.5, ha="right", va="bottom", family="monospace",
                    bbox=dict(fc="white", ec="0.8", lw=0.5, pad=2.5, alpha=0.85))

        # bottom: residuals
        ax_bot.errorbar(true_v, res_v, yerr=[elo, ehi], **pt_kw)
        ax_bot.axhline(0, color="k", lw=0.8, ls="--", alpha=0.7)
        ax_bot.axhspan(-medae, medae, color=ORANGE, alpha=0.12)
        ax_bot.set_xlabel(xlabel, labelpad=3)
        ax_bot.set_ylabel(r"$\Delta$" if pi == 0 else "", labelpad=3)
        ax_bot.tick_params(direction="in", top=True, right=True)
        ax_bot.grid(True, lw=0.4, alpha=0.4)
        rlim = 1.5 * np.percentile(np.abs(res_v), 95)
        ax_bot.set_ylim(-rlim, rlim)

    return fig


print("Recovery helpers ready.")


In [ ]:
# ── Run recovery for all four single-component models ─────────────────────────
# Each model runs ~200 simulations + 200×1000 posterior samples.
# On CPU this takes a few minutes per model; on GPU ~30 s.

np.random.seed(SEED_R)
torch.manual_seed(SEED_R)

for model_type in MODEL_ORDER:
    pt = MODELS_DIR / f"posterior_{model_type}_n1.pt"
    if not pt.exists():
        print(f"[skip] {pt.name}")
        continue
    print(f"\n=== {model_type} ===")

    posterior, meta = load_posterior(pt, device=DEVICE)
    input_channels  = int(meta.get("input_channels", 2))
    n_freq          = int(meta.get("n_freq", 127))
    lambda_sq       = np.array(meta["lambda_sq"])
    pb              = meta["prior_bounds"]
    lo, hi          = np.array(pb["low"]), np.array(pb["high"])
    print(f"  prior lo={lo.round(2)}  hi={hi.round(2)}")

    freq_file  = make_freq_file(lambda_sq)
    weights_1d = np.ones(n_freq, dtype=np.float32)
    thetas     = sobol_prior(lo, hi, N_TEST)
    sim        = RMSimulator(freq_file, n_components=1, model_type=model_type)
    sigma_bases = np.random.uniform(SIGMA_MIN_R, SIGMA_MAX_R, N_TEST).astype(np.float32)

    # simulate spectra
    obs = []
    for i in range(N_TEST):
        qu = sim.simulate_batch(thetas[i:i+1], weights_1d[None], noise_sigma=sigma_bases[i])[0]
        obs.append(qu)
    obs = np.array(obs, dtype=np.float32)            # (N_TEST, 2*n_freq)
    print(f"  Simulated {N_TEST} spectra.")

    # run inference
    all_samples = []
    for i in range(N_TEST):
        x = obs[i]
        if input_channels == 3:
            x = np.concatenate([x, weights_1d])
        x_t = torch.tensor(x, dtype=torch.float32, device=DEVICE)
        s = posterior.sample((N_POST_R,), x=x_t).cpu().numpy()
        all_samples.append(s)
        if (i + 1) % 50 == 0:
            print(f"  inference {i+1}/{N_TEST}")
    all_samples = np.array(all_samples)               # (N_TEST, N_POST_R, n_params)

    # plot and save
    fig = make_recovery_figure(model_type, thetas, all_samples, lo, hi)
    out = IMAGES_DIR / f"posterior_{model_type}_n1_recovery.pdf"
    fig.savefig(out, bbox_inches="tight", dpi=200)
    plt.show()
    print(f"  Saved → {out}")


---
## 3. Two-Component Model Validation

The `faraday_thin_n2` model fits **two** Faraday-thin screens simultaneously.
Each screen has its own `(φ, p₀, χ₀)`, giving **6 parameters** total.

### Label ordering convention
During training, components are always sorted so that **RM₁ > RM₂** (descending).
At inference time, `sort_components_by_rm()` re-sorts posterior samples the same way,
so the label assignments are consistent.

### Figure 3a — Recovery plot (`images/n2_recovery.pdf`)
Same layout as the single-component case but with 6 columns.

### Figure 3b — Single-example posterior (`images/n2_case2.pdf`)
A representative case with ΔRM = 59.5 rad m⁻².
Left column: Q and U spectra with posterior-predictive draws.
Right column: 6×6 corner plot of all posterior parameters.


In [ ]:
# ── N=2 parameter recovery ────────────────────────────────────────────────────
# Reproduces: images/n2_recovery.pdf (scripts/validate_n2.py)

from src.simulator.prior import sort_components_by_rm

MODEL_PATH_N2 = MODELS_DIR / "posterior_faraday_thin_n2.pt"

if not MODEL_PATH_N2.exists():
    print(f"[skip] {MODEL_PATH_N2.name} not found")
else:
    N_TEST_N2    = 256
    N_POST_N2    = 1000
    N_COMP       = 2
    PARAMS_PER_C = 3

    PARAM_LABELS_N2 = [
        r"$\phi_1$", r"$p_1$", r"$\chi_{0,1}$",
        r"$\phi_2$", r"$p_2$", r"$\chi_{0,2}$",
    ]
    PARAM_UNITS_N2 = [r"rad m$^{-2}$", "", "rad", r"rad m$^{-2}$", "", "rad"]

    np.random.seed(0); torch.manual_seed(0)

    posterior_n2, meta_n2 = load_posterior(MODEL_PATH_N2, device=DEVICE)
    ic_n2     = int(meta_n2.get("input_channels", 3))
    n_freq_n2 = int(meta_n2.get("n_freq", 128))
    lsq_n2    = np.array(meta_n2["lambda_sq"])
    lo_n2     = np.array(meta_n2["prior_bounds"]["low"])
    hi_n2     = np.array(meta_n2["prior_bounds"]["high"])

    freq_file_n2  = make_freq_file(lsq_n2, "/tmp/vroom_n2_freqs.txt")
    weights_n2    = np.ones(n_freq_n2, dtype=np.float32)
    thetas_raw    = sobol_prior(lo_n2, hi_n2, N_TEST_N2)
    thetas_n2     = sort_components_by_rm(thetas_raw, N_COMP, PARAMS_PER_C)
    sim_n2        = RMSimulator(freq_file_n2, n_components=N_COMP, model_type="faraday_thin")
    sigma_bases_n2 = np.random.uniform(SIGMA_MIN_R, SIGMA_MAX_R, N_TEST_N2).astype(np.float32)

    obs_n2 = []
    for i in range(N_TEST_N2):
        qu = sim_n2.simulate_batch(thetas_n2[i:i+1], weights_n2[None],
                                    noise_sigma=sigma_bases_n2[i])[0]
        obs_n2.append(qu)
    obs_n2 = np.array(obs_n2, dtype=np.float32)
    print(f"Simulated {N_TEST_N2} two-component spectra.")

    samples_n2 = []
    for i in range(N_TEST_N2):
        x = obs_n2[i]
        if ic_n2 == 3:
            x = np.concatenate([x, weights_n2])
        x_t = torch.tensor(x, dtype=torch.float32, device=DEVICE)
        s = posterior_n2.sample((N_POST_N2,), x=x_t).cpu().numpy()
        s = sort_components_by_rm(s, N_COMP, PARAMS_PER_C)
        samples_n2.append(s)
        if (i + 1) % 64 == 0:
            print(f"  {i+1}/{N_TEST_N2}")
    samples_n2 = np.array(samples_n2)   # (N_TEST_N2, N_POST_N2, 6)

    # print coverage
    print("\nCoverage:")
    for pi, lbl in enumerate(PARAM_LABELS_N2):
        c68 = coverage_at_68(samples_n2[:, :, pi], thetas_n2[:, pi])
        print(f"  {lbl}: Cov68={c68:.0%}")

    # plot
    n_p  = thetas_n2.shape[1]
    lo_n2_arr, hi_n2_arr = np.array(lo_n2), np.array(hi_n2)
    med_n2    = np.median(samples_n2, axis=1)
    p16_n2    = np.percentile(samples_n2, 16, axis=1)
    p84_n2    = np.percentile(samples_n2, 84, axis=1)
    err_lo_n2 = med_n2 - p16_n2
    err_hi_n2 = p84_n2 - med_n2
    resid_n2  = med_n2 - thetas_n2

    fig = plt.figure(figsize=(n_p * 2.2, 5.2))
    fig.suptitle("2-component Faraday-thin — parameter recovery",
                 fontsize=10, fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(2, n_p, figure=fig, hspace=0.10, wspace=0.38,
                           top=0.91, bottom=0.12, left=0.08, right=0.99)
    gs.set_height_ratios([2.6, 1.0])
    pt_kw = dict(fmt="o", ms=1.8, alpha=0.55, color=ORANGE,
                 ecolor=ORANGE, elinewidth=0.5, capsize=0, capthick=0)

    for pi in range(n_p):
        ax_t = fig.add_subplot(gs[0, pi])
        ax_b = fig.add_subplot(gs[1, pi], sharex=ax_t)
        tv = thetas_n2[:, pi]; mv = med_n2[:, pi]; rv = resid_n2[:, pi]
        medae = float(np.median(np.abs(rv)))
        bias  = float(np.mean(rv))
        cov68 = coverage_at_68(samples_n2[:, :, pi], tv)
        pad  = 0.04 * (hi_n2_arr[pi] - lo_n2_arr[pi])
        lim  = [lo_n2_arr[pi] - pad, hi_n2_arr[pi] + pad]

        ax_t.errorbar(tv, mv, yerr=[err_lo_n2[:, pi], err_hi_n2[:, pi]], **pt_kw)
        ax_t.plot(lim, lim, "k--", lw=0.8, alpha=0.7)
        ax_t.set_xlim(lim); ax_t.set_ylim(lim)
        ax_t.set_ylabel("Recovered" if pi == 0 else "", labelpad=3)
        ax_t.tick_params(labelbottom=False, direction="in", top=True, right=True)
        ax_t.set_title(PARAM_LABELS_N2[pi], pad=3)
        ax_t.grid(True, lw=0.4, alpha=0.4)
        ax_t.text(0.97, 0.04,
                  f"MedAE={medae:.2g}\nBias={bias:+.2g}\nCov={cov68:.0%}",
                  transform=ax_t.transAxes, fontsize=6.5, ha="right", va="bottom",
                  family="monospace",
                  bbox=dict(fc="white", ec="0.8", lw=0.5, pad=2.5, alpha=0.85))

        unit_str = PARAM_UNITS_N2[pi]
        xlabel   = PARAM_LABELS_N2[pi] + (f" ({unit_str})" if unit_str else "")
        ax_b.errorbar(tv, rv, yerr=[err_lo_n2[:, pi], err_hi_n2[:, pi]], **pt_kw)
        ax_b.axhline(0, color="k", lw=0.8, ls="--", alpha=0.7)
        ax_b.axhspan(-medae, medae, color=ORANGE, alpha=0.12)
        ax_b.set_xlabel(xlabel, labelpad=3)
        ax_b.set_ylabel(r"$\Delta$" if pi == 0 else "", labelpad=3)
        ax_b.tick_params(direction="in", top=True, right=True)
        ax_b.grid(True, lw=0.4, alpha=0.4)
        rlim = 1.5 * np.percentile(np.abs(rv), 95)
        ax_b.set_ylim(-rlim, rlim)

    fig.savefig(IMAGES_DIR / "n2_recovery.pdf", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved → {IMAGES_DIR / 'n2_recovery.pdf'}")


In [ ]:
# ── N=2 single-example demo — ΔRM = 59.5 rad/m² ──────────────────────────────
# Reproduces: images/n2_case2.pdf (scripts/plot_n2_case2.py)
#
# A representative two-component case with a large RM separation.
# Left panels: Q(λ²) and U(λ²) spectra with posterior-predictive draws (blue/green).
# Right panel: 6×6 corner plot — marginal histograms on diagonal, KDE contours off-diagonal.

if not MODEL_PATH_N2.exists():
    print("[skip] n2 model not found")
else:
    # Ground-truth parameters: [phi_1, p_1, chi0_1, phi_2, p_2, chi0_2]
    # phi are in rad/m², p dimensionless, chi0 in radians
    # Components sorted RM-descending: phi_1=15 > phi_2=-44.5
    THETA_TRUE_C2 = np.array([15.0, 0.793, 0.216, -44.5, 0.818, 0.231], dtype=np.float32)
    NOISE_C2      = 0.015
    N_SAMPLES_C2  = 2000
    N_PP          = 80       # posterior-predictive draws for spectrum panels

    np.random.seed(42); torch.manual_seed(42)

    N_PARAMS_C2   = 6
    PARAM_LABS_C2 = [
        r"$\phi_1$",   r"$p_1$",   r"$\chi_{0,1}$",
        r"$\phi_2$",   r"$p_2$",   r"$\chi_{0,2}$",
    ]

    lsq_c2   = np.array(meta_n2["lambda_sq"])
    ic_c2    = int(meta_n2.get("input_channels", 3))
    nf_c2    = int(meta_n2.get("n_freq", 128))
    wts_c2   = np.ones(nf_c2, dtype=np.float32)
    sim_c2   = sim_n2   # reuse simulator from previous cell

    qu_true  = sim_c2.simulate_noiseless(THETA_TRUE_C2[None])
    Q_true   = qu_true[:nf_c2]
    U_true   = qu_true[nf_c2:]

    qu_obs   = sim_c2.simulate_batch(THETA_TRUE_C2[None], wts_c2[None],
                                      noise_sigma=NOISE_C2)[0]
    Q_obs, U_obs = qu_obs[:nf_c2], qu_obs[nf_c2:]

    x_in = np.concatenate([Q_obs, U_obs])
    if ic_c2 == 3:
        x_in = np.concatenate([x_in, wts_c2])
    x_t = torch.tensor(x_in, dtype=torch.float32, device=DEVICE)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        samples_c2 = posterior_n2.sample((N_SAMPLES_C2,), x=x_t).cpu().numpy()
    samples_c2 = sort_components_by_rm(samples_c2, N_COMP, PARAMS_PER_C)

    print("Truth :", THETA_TRUE_C2)
    print("Median:", np.median(samples_c2, axis=0).round(3))

    idx_pp = np.random.choice(N_SAMPLES_C2, N_PP, replace=False)

    # ── Figure: Q/U panels (left) + 6×6 corner (right) ───────────────────────
    fig = plt.figure(figsize=(15, 8))

    # Left column: Q and U
    ax_q = fig.add_axes([0.04, 0.52, 0.28, 0.40])
    ax_u = fig.add_axes([0.04, 0.08, 0.28, 0.40])

    # Right column: 6×6 corner grid
    cl, cb, cw, ch = 0.36, 0.06, 0.62, 0.91
    gap    = 0.004
    cell_w = (cw - (N_PARAMS_C2 - 1) * gap) / N_PARAMS_C2
    cell_h = (ch - (N_PARAMS_C2 - 1) * gap) / N_PARAMS_C2
    axes_grid = np.empty((N_PARAMS_C2, N_PARAMS_C2), dtype=object)
    for row in range(N_PARAMS_C2):
        for col in range(N_PARAMS_C2):
            x0 = cl + col * (cell_w + gap)
            y0 = cb + (N_PARAMS_C2 - 1 - row) * (cell_h + gap)
            axes_grid[row, col] = fig.add_axes([x0, y0, cell_w, cell_h])

    # Q spectrum
    for k in idx_pp:
        qu_pp = sim_c2.simulate_noiseless(samples_c2[k:k+1])
        ax_q.plot(lsq_c2, qu_pp[:nf_c2], color=BLUE, alpha=0.05, lw=0.5, zorder=1)
    ax_q.scatter(lsq_c2, Q_obs, s=3, color=GREY, alpha=0.6, zorder=2, label="Obs")
    ax_q.plot(lsq_c2, Q_true, color=RED, lw=1.2, zorder=3, label="Truth")
    dRM = float(THETA_TRUE_C2[0] - THETA_TRUE_C2[3])
    ax_q.set_title(
        rf"$\phi_1={THETA_TRUE_C2[0]:.0f}$, $p_1={THETA_TRUE_C2[1]:.2f}$, "
        rf"$\phi_2={THETA_TRUE_C2[3]:.0f}$, $p_2={THETA_TRUE_C2[4]:.2f}$   "
        rf"($\Delta\phi={dRM:.1f}$ rad m$^{{-2}}$)",
        fontsize=8, pad=3,
    )
    ax_q.set_ylabel(r"$Q$")
    ax_q.tick_params(labelbottom=False, direction="in", top=True, right=True)
    ax_q.grid(True, lw=0.35, alpha=0.4)
    ax_q.legend(fontsize=7.5, framealpha=0.85, loc="upper right")

    # U spectrum
    for k in idx_pp:
        qu_pp = sim_c2.simulate_noiseless(samples_c2[k:k+1])
        ax_u.plot(lsq_c2, qu_pp[nf_c2:], color=GREEN, alpha=0.05, lw=0.5, zorder=1)
    ax_u.scatter(lsq_c2, U_obs, s=3, color=GREY, alpha=0.6, zorder=2)
    ax_u.plot(lsq_c2, U_true, color=GREEN, lw=1.2, zorder=3)
    ax_u.set_xlabel(r"$\lambda^2\;(\mathrm{m}^2)$")
    ax_u.set_ylabel(r"$U$")
    ax_u.tick_params(direction="in", top=True, right=True)
    ax_u.grid(True, lw=0.35, alpha=0.4)

    # Corner plot
    draw_corner(axes_grid, samples_c2, THETA_TRUE_C2, N_PARAMS_C2, PARAM_LABS_C2)
    axes_grid[0, 0].set_title("Posterior", pad=4, fontsize=9)

    fig.savefig(IMAGES_DIR / "n2_case2.pdf", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved → {IMAGES_DIR / 'n2_case2.pdf'}")


---
## 4. Real Data: MACS J1752+4440

**⚠️ Tier 3** — The next two cells require the G71 FITS cubes and pre-computed result maps.
They will skip gracefully if the data is absent.

MACS J1752+4440 (nicknamed 'G71' in this work) is a galaxy cluster observed with the VLA
as part of the LOFAR ELDF survey. The Q, U, and I cubes have been normalised
to fractional polarisation (Q/I, U/I) before being passed to VROOM-SBI.

### §4.3 Spectral index map
Spectral index α (where F∝ν^α) mapped across the cluster from VROOM-SBI's
separate spectral-shape model.  The output FITS maps were produced by running
`python scripts/infer_spectral_cube.py` (see CLI documentation).

**Required files** (relative to repo root):
- `outputs/g71_spectral/alpha_mean.fits`
- `outputs/g71_spectral/alpha_std.fits`

### §4.4 RM maps and comparison
RM synthesis peak-RM map vs. VROOM-SBI posterior mean RM, and a pixel-by-pixel
scatter comparison against RMtools-QUfit.

**Required files** (relative to `$G71_DIR`):
- `comparison_table.csv`
- `p_map.fits`
- `results_vroom/out_peak_rm.fits`
- `results_sobol/rm_mean_comp1.fits`, `rm_std_comp1.fits`


In [ ]:
# ── §4.3 Spectral index map ───────────────────────────────────────────────────
# Reproduces: images/spectral_index_map.pdf (scripts/plot_spectral_index.py)
# ⚠️ TIER 3 — requires outputs/g71_spectral/alpha_mean.fits etc.

ALPHA_F = REPO / "outputs" / "g71_spectral" / "alpha_mean.fits"
STD_F   = REPO / "outputs" / "g71_spectral" / "alpha_std.fits"

if not ALPHA_F.exists() or not STD_F.exists():
    print("[skip] spectral index maps not found — run the spectral cube inference first.")
    print(f"  Expected: {ALPHA_F}")
else:
    from astropy.io import fits as _fits
    from astropy.wcs import WCS

    hdu   = _fits.open(ALPHA_F)[0]
    alpha = hdu.data.squeeze().astype(np.float32)
    wcs   = WCS(hdu.header)
    if wcs.naxis > 2:
        wcs = wcs.celestial

    std  = _fits.open(STD_F)[0].data.squeeze().astype(np.float32)
    mask = np.isfinite(alpha) & (alpha != 0) & np.isfinite(std) & (std != 0)
    alpha = np.where(mask, alpha, np.nan)
    std   = np.where(mask, std,   np.nan)

    vmin_a, vmax_a = -2.0, -0.3
    vmin_s, vmax_s =  0.22,  0.50

    fig = plt.figure(figsize=(8.0, 4.2))
    ax1  = fig.add_axes([0.07, 0.12, 0.38, 0.80], projection=wcs)
    ax2  = fig.add_axes([0.54, 0.12, 0.38, 0.80], projection=wcs)
    cax1 = fig.add_axes([0.46, 0.12, 0.015, 0.80])
    cax2 = fig.add_axes([0.93, 0.12, 0.015, 0.80])

    cmap_alpha = plt.get_cmap("RdYlBu_r").copy(); cmap_alpha.set_bad("none")
    cmap_std   = plt.get_cmap("magma").copy();     cmap_std.set_bad("none")

    im1 = ax1.imshow(alpha, origin="lower", cmap=cmap_alpha,
                     vmin=vmin_a, vmax=vmax_a, interpolation="nearest", aspect="equal")
    im2 = ax2.imshow(std,   origin="lower", cmap=cmap_std,
                     vmin=vmin_s, vmax=vmax_s, interpolation="nearest", aspect="equal")

    cb1 = fig.colorbar(im1, cax=cax1); cb1.set_label(r"$\alpha$", labelpad=4)
    cb2 = fig.colorbar(im2, cax=cax2); cb2.set_label(r"$\sigma_\alpha$", labelpad=4)

    for ax, title in [(ax1, r"Spectral index $\alpha$"),
                      (ax2, r"Uncertainty $\sigma_\alpha$")]:
        ra  = ax.coords["ra"]
        dec = ax.coords["dec"]
        ra.set_axislabel("RA (J2000)", minpad=0.5)
        dec.set_axislabel("Dec (J2000)", minpad=0.3)
        ra.set_major_formatter("hh:mm:ss")
        dec.set_major_formatter("dd:mm")
        ra.set_ticks(number=4); dec.set_ticks(number=4)
        ax.set_title(title, pad=4, fontsize=9, fontweight="bold")
        ax.coords.grid(color="white", alpha=0.15, lw=0.4)

    ax2.coords["dec"].set_axislabel("")
    ax2.coords["dec"].set_ticklabel_visible(False)

    fig.savefig(IMAGES_DIR / "spectral_index_map.pdf", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved → {IMAGES_DIR / 'spectral_index_map.pdf'}")


In [ ]:
# ── §4.4 RM maps 2×2 ─────────────────────────────────────────────────────────
# Reproduces: images/rm_maps_2x2.pdf (scripts/plot_rm_maps_2x2.py)
# ⚠️ TIER 3 — requires G71 result FITS maps.

COMP_CSV  = G71_DIR / "comparison_table.csv"
P_MAP_F   = G71_DIR / "p_map.fits"
PEAK_RM_F = G71_DIR / "results_vroom" / "out_peak_rm.fits"
SBI_RM_F  = G71_DIR / "results_sobol" / "rm_mean_comp1.fits"
SBI_STD_F = G71_DIR / "results_sobol" / "rm_std_comp1.fits"

_required = [COMP_CSV, P_MAP_F, PEAK_RM_F, SBI_RM_F, SBI_STD_F]
if not all(f.exists() for f in _required):
    missing = [str(f) for f in _required if not f.exists()]
    print("[skip] Missing required G71 files:")
    for m in missing:
        print(f"  {m}")
else:
    import pandas as pd
    from astropy.io import fits as _fits
    from astropy.wcs import WCS
    import matplotlib.colors as mcolors

    df       = pd.read_csv(COMP_CSV)
    p_hdu    = _fits.open(P_MAP_F)[0]
    p_map    = p_hdu.data.squeeze().astype(np.float32)
    wcs_p    = WCS(p_hdu.header).celestial

    rm_syn   = _fits.getdata(PEAK_RM_F).squeeze().astype(np.float32)
    rm_sbi   = _fits.getdata(SBI_RM_F).squeeze().astype(np.float32)
    rm_std   = _fits.getdata(SBI_STD_F).squeeze().astype(np.float32)
    resid    = rm_sbi - rm_syn

    # shared colour limits
    vmin_rm  = np.nanpercentile(rm_syn,  2)
    vmax_rm  = np.nanpercentile(rm_syn, 98)
    vmax_std = np.nanpercentile(rm_std, 98)
    vmax_res = np.nanpercentile(np.abs(resid), 98)

    # ── 2×2 layout ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        2, 2, figsize=(10, 8),
        subplot_kw={"projection": wcs_p},
        gridspec_kw={"hspace": 0.05, "wspace": 0.30},
    )

    panel_data   = [rm_syn,  rm_sbi,  resid,             rm_std]
    panel_titles = ["(a) RM synthesis (peak)",
                    "(b) VROOM-SBI RM (mean)",
                    "(c) Residual SBI − synthesis",
                    "(d) VROOM-SBI RM uncertainty"]
    panel_vmins  = [vmin_rm, vmin_rm, -vmax_res, 0]
    panel_vmaxs  = [vmax_rm, vmax_rm,  vmax_res, vmax_std]
    panel_cmaps  = ["RdBu_r", "RdBu_r", "coolwarm", "magma"]

    for (r, c), ax, data, title, vmi, vma, cmap in zip(
        [(0,0),(0,1),(1,0),(1,1)],
        [axes[0,0], axes[0,1], axes[1,0], axes[1,1]],
        panel_data, panel_titles, panel_vmins, panel_vmaxs, panel_cmaps,
    ):
        im = ax.imshow(data, origin="lower", cmap=cmap,
                       vmin=vmi, vmax=vma, interpolation="nearest")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                     label=r"RM (rad m$^{-2}$)" if "uncertainty" not in title else r"$\sigma_{RM}$ (rad m$^{-2}$)")
        ax.set_title(title, pad=4, fontsize=9)
        ax.coords.grid(color="white", alpha=0.15, lw=0.4)
        ax.coords["ra"].set_axislabel("RA (J2000)" if r == 1 else "")
        ax.coords["dec"].set_axislabel("Dec (J2000)" if c == 0 else "")
        ax.coords["ra"].set_major_formatter("hh:mm:ss")
        ax.coords["dec"].set_major_formatter("dd:mm")

        # overplot QUfit pixels on residual panel
        if "Residual" in title and "ra" in df.columns:
            pix = wcs_p.all_world2pix(
                df[["ra", "dec"]].values[:100], 0
            )
            ax.scatter(pix[:, 0], pix[:, 1],
                       s=4, c="white", alpha=0.6, linewidths=0, zorder=5)

    fig.savefig(IMAGES_DIR / "rm_maps_2x2.pdf", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved → {IMAGES_DIR / 'rm_maps_2x2.pdf'}")


In [ ]:
# ── §4.4 Pixel-by-pixel RM comparison: VROOM-SBI vs QUfit ─────────────────────
# Reproduces: images/comparison_n1_sobol.pdf (scripts/plot_comparison_n1_200.py)
# ⚠️ TIER 3

COMP_CSV2 = G71_DIR / "comparison_table.csv"
SBI_RM2   = G71_DIR / "results_sobol" / "rm_mean_comp1.fits"
SBI_P16   = G71_DIR / "results_sobol" / "rm_p16_comp1.fits"
SBI_P84   = G71_DIR / "results_sobol" / "rm_p84_comp1.fits"

_req2 = [COMP_CSV2, SBI_RM2, SBI_P16, SBI_P84]
if not all(f.exists() for f in _req2):
    print("[skip] G71 comparison files not found.")
else:
    import pandas as pd
    from astropy.io import fits as _fits

    df2      = pd.read_csv(COMP_CSV2)
    sbi_rm   = _fits.getdata(SBI_RM2).squeeze()
    sbi_p16  = _fits.getdata(SBI_P16).squeeze()
    sbi_p84  = _fits.getdata(SBI_P84).squeeze()

    # extract values at comparison pixel locations
    rows, cols   = df2["pix_row"].values, df2["pix_col"].values
    rm_qufit     = df2["rm_qufit"].values
    rm_qufit_err = df2["rm_qufit_err"].values
    rm_sbi_pix   = sbi_rm[rows, cols]
    err_lo_pix   = rm_sbi_pix - sbi_p16[rows, cols]
    err_hi_pix   = sbi_p84[rows, cols] - rm_sbi_pix
    frac_pol     = df2["frac_pol"].values if "frac_pol" in df2.columns else np.ones(len(df2))

    fig, axes = plt.subplots(2, 1, figsize=(6, 8), sharex=False,
                             gridspec_kw={"hspace": 0.35})

    for ax, x_vals, x_err, x_label in [
        (axes[0], rm_qufit, rm_qufit_err, r"QUfit RM (rad m$^{-2}$)"),
        (axes[1], rm_sbi_pix, [err_lo_pix, err_hi_pix], r"VROOM-SBI RM (rad m$^{-2}$)"),
    ]:
        sc = ax.scatter(rm_qufit, rm_sbi_pix, c=frac_pol, cmap="viridis",
                        s=8, alpha=0.6, linewidths=0, zorder=3)
        plt.colorbar(sc, ax=ax, label="Fractional polarisation")
        lim = [
            min(rm_qufit.min(), rm_sbi_pix.min()) - 5,
            max(rm_qufit.max(), rm_sbi_pix.max()) + 5,
        ]
        ax.plot(lim, lim, "k--", lw=0.8, alpha=0.7, label="1:1")
        ax.set_xlabel(r"QUfit RM (rad m$^{-2}$)", labelpad=3)
        ax.set_ylabel(r"VROOM-SBI RM (rad m$^{-2}$)", labelpad=3)
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.tick_params(direction="in", top=True, right=True)
        ax.grid(True, lw=0.4, alpha=0.4)

    fig.suptitle("MACS J1752+4440: VROOM-SBI vs QUfit RM (N=1 Faraday-thin)",
                 fontsize=9, fontweight="bold")
    fig.savefig(IMAGES_DIR / "comparison_n1_sobol.pdf", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved → {IMAGES_DIR / 'comparison_n1_sobol.pdf'}")


---
## 5. VROOM-SBI vs RMtools-QUfit: Posterior Comparison

**⚠️ Tier 2** — These cells need:
- `test_sobol/posterior_faraday_thin_n1.pt` (a model variant, may fall back to `models/`)
- Cached QUfit posteriors: `scripts/qufit_cache_{row}_{col}.npz`
- For the G71 comparison: the G71 cubes (Tier 3)

### What these figures show

Corner plots overlaying the VROOM-SBI posterior (blue) and the RMtools-QUfit
nested-sampling posterior (red) for identical data.

**Figure: `corner_compare_plots/sim_test_sobol.pdf`** — simulated test spectrum  
  (truth = RM=25, p=0.30, χ₀=0.5 rad). Both methods see the same synthetic data.

**Figure: `corner_compare_plots/G71_sobol.pdf`** — three representative pixels  
  from MACS J1752+4440 (best-agreement pixels).

VROOM-SBI is 100–1000× faster than nested sampling while producing nearly identical posteriors.


In [ ]:
# ── Corner comparison: simulated test spectrum ────────────────────────────────
# Reproduces: corner_compare_plots/sim_test_sobol.pdf (scripts/corner_compare.py)
#
# We simulate one spectrum with known truth (RM=25, p=0.30, chi0=0.5 rad)
# and compare the VROOM-SBI posterior to a cached QUfit posterior.
#
# chi0 in VROOM-SBI is in radians; QUfit reports in degrees — we convert.

FREQ_G71  = G71_DIR / "freq_regrid_clean.txt"
NOISE_G71 = G71_DIR / "noise_per_chan.fits"

# Try the sobol model variant, fall back to main models dir
_sobol_pt = REPO / "test_sobol" / "posterior_faraday_thin_n1.pt"
_main_pt  = MODELS_DIR / "posterior_faraday_thin_n1.pt"
CORNER_PT = _sobol_pt if _sobol_pt.exists() else _main_pt

if not CORNER_PT.exists():
    print("[skip] No faraday_thin_n1 model found.")
elif not FREQ_G71.exists():
    print("[skip] G71 frequency file not found — cannot build simulation on G71 grid.")
    print("  Using model's own lambda_sq grid as fallback for the simulated test.")
    _use_g71_grid = False
else:
    _use_g71_grid = True

if CORNER_PT.exists():
    SIM_RM   = 25.0
    SIM_AMP  = 0.30
    SIM_CHI0 = 0.5    # radians
    N_SBI    = 5000

    LABS_CC = [r"RM  (rad m$^{-2}$)", r"frac. pol. $p$", r"$\psi_0$  (deg)"]

    np.random.seed(0); torch.manual_seed(0)

    posterior_cc, meta_cc = load_posterior(CORNER_PT, device=DEVICE)
    ic_cc  = int(meta_cc.get("input_channels", 2))
    nf_cc  = int(meta_cc.get("n_freq", 127))
    lsq_cc = np.array(meta_cc["lambda_sq"])

    if _use_g71_grid if 'FREQ_G71' in dir() and FREQ_G71.exists() else False:
        freq_hz_cc = np.loadtxt(FREQ_G71)
        lsq_cc_sim = (3e8 / freq_hz_cc) ** 2
        freq_file_cc = str(FREQ_G71)
        from astropy.io import fits as _fits
        noise_cc = _fits.getdata(NOISE_G71).squeeze().astype(np.float32)
        weights_cc = compute_weights(noise_cc)
        nf_cc = len(freq_hz_cc)
    else:
        freq_file_cc = make_freq_file(lsq_cc, "/tmp/vroom_corner_freqs.txt")
        weights_cc   = np.ones(nf_cc, dtype=np.float32)
        lsq_cc_sim   = lsq_cc

    sim_cc     = RMSimulator(freq_file_cc, n_components=1, model_type="faraday_thin")
    theta_sim  = np.array([[SIM_RM, SIM_AMP, SIM_CHI0]])
    sigma_sim  = SIM_AMP * 0.05
    qu_sim     = sim_cc.simulate_batch(theta_sim, weights_cc[None], noise_sigma=sigma_sim)[0]

    # VROOM-SBI posterior
    x_cc = qu_sim
    if ic_cc == 3:
        x_cc = np.concatenate([x_cc, weights_cc])
    x_t_cc = torch.tensor(x_cc, dtype=torch.float32, device=DEVICE)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sbi_samples = posterior_cc.sample((N_SBI,), x=x_t_cc).cpu().numpy()

    # sbi_samples: [RM, p, chi0_rad] — convert chi0 to degrees for comparison
    sbi_arr = sbi_samples.copy()
    sbi_arr[:, 2] = np.degrees(sbi_arr[:, 2])
    truth_cc = [SIM_RM, SIM_AMP, float(np.degrees(SIM_CHI0))]

    # Load or note QUfit cache
    cache_sim = REPO / "scripts" / "qufit_cache_sim.npz"
    if cache_sim.exists():
        c = np.load(cache_sim)
        rm_dict_sim = {"RM": c["rm"], "amp": c["amp"], "chi0_deg": c["chi0_deg"]}
        rm_arr_sim  = np.column_stack([rm_dict_sim["RM"], rm_dict_sim["amp"],
                                        rm_dict_sim["chi0_deg"]])
        arrays_cc   = [sbi_arr, rm_arr_sim]
    else:
        print("[note] No qufit_cache_sim.npz — showing VROOM-SBI only.")
        arrays_cc = [sbi_arr]

    # corner plot
    N_P_CC = 3
    all_vals = np.vstack(arrays_cc)
    ranges_cc = [_get_range(all_vals[:, i]) for i in range(N_P_CC)]

    fig, axes_cc = plt.subplots(N_P_CC, N_P_CC, figsize=(7, 7))
    fig.subplots_adjust(hspace=0.05, wspace=0.05)
    fig.suptitle(f"Simulated test: RM={SIM_RM}, p={SIM_AMP}, χ₀={np.degrees(SIM_CHI0):.1f}°",
                 fontsize=10, fontweight="bold")

    colors_cc = [SBI_COL, RM_COL]
    labels_cc = ["VROOM-SBI", "RMtools-QUfit"]

    for row in range(N_P_CC):
        for col in range(N_P_CC):
            ax = axes_cc[row, col]
            if col > row:
                ax.set_visible(False); continue
            if row == col:
                for arr, clr in zip(arrays_cc, colors_cc):
                    vals = arr[:, row]
                    ax.hist(vals, bins=40, range=ranges_cc[row],
                            color=clr, alpha=0.35, density=True, linewidth=0)
                    kde = gaussian_kde(vals, bw_method=0.3)
                    xp  = np.linspace(*ranges_cc[row], 300)
                    ax.plot(xp, kde(xp), color=clr, lw=1.8)
                ax.axvline(truth_cc[row], color=TRUTH_COL, lw=1.8, ls="--")
                ax.set_xlim(ranges_cc[row]); ax.set_yticks([])
            else:
                for arr, clr in zip(arrays_cc, colors_cc):
                    _draw_kde_contours(ax, arr[:, col], arr[:, row],
                                       ranges_cc[col], ranges_cc[row], color=clr)
                ax.axvline(truth_cc[col], color=TRUTH_COL, lw=1.0, ls="--", alpha=0.7)
                ax.axhline(truth_cc[row], color=TRUTH_COL, lw=1.0, ls="--", alpha=0.7)
                ax.set_xlim(ranges_cc[col]); ax.set_ylim(ranges_cc[row])

            if row == N_P_CC - 1:
                ax.set_xlabel(LABS_CC[col], fontsize=9)
            else:
                ax.set_xticklabels([])
            if col == 0 and row != 0:
                ax.set_ylabel(LABS_CC[row], fontsize=9)
            else:
                ax.set_yticklabels([])
            ax.tick_params(labelsize=8)

    patches = [mpatches.Patch(color=c, alpha=0.8, label=l)
               for c, l in zip(colors_cc, labels_cc)]
    patches.append(plt.Line2D([0], [0], color=TRUTH_COL, lw=1.8, ls="--", label="Truth"))
    axes_cc[0, 0].legend(handles=patches, fontsize=8, loc="upper left")

    out_cc = REPO / "corner_compare_plots"
    out_cc.mkdir(exist_ok=True)
    fig.savefig(out_cc / "sim_test_sobol.pdf", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved → {out_cc / 'sim_test_sobol.pdf'}")


In [ ]:
# ── Corner comparison: three best-agreement G71 pixels ────────────────────────
# Reproduces: corner_compare_plots/G71_sobol.pdf (scripts/corner_compare.py)
# ⚠️ TIER 2/3 — needs G71 cubes + qufit_cache_*.npz
#
# Three best-agreement pixels selected from the cluster as representative
# cases where QUfit and VROOM-SBI posteriors are compared side by side.

G71_PIXELS = [(2551, 1979), (2059, 2593), (2037, 2562)]

CUBE_Q = G71_DIR / "cube_Q_regrid.fits"
CUBE_U = G71_DIR / "cube_U_regrid.fits"
CUBE_I = G71_DIR / "cube_I_regrid.fits"

if not CORNER_PT.exists():
    print("[skip] No model found.")
else:
    # Check if we have qufit caches for the G71 pixels
    cache_dir = REPO / "scripts"
    cached = {(r, c): cache_dir / f"qufit_cache_{r}_{c}.npz" for r, c in G71_PIXELS}
    have_cache = {k: v.exists() for k, v in cached.items()}
    have_cubes = CUBE_Q.exists() and CUBE_U.exists() and CUBE_I.exists()

    if not any(have_cache.values()) and not have_cubes:
        print("[skip] No qufit_cache_*.npz files and no G71 cubes found.")
        print("  Run scripts/corner_compare.py once to generate the caches.")
    else:
        from astropy.io import fits as _fits

        if have_cubes:
            cube_Q = _fits.getdata(CUBE_Q).squeeze()
            cube_U = _fits.getdata(CUBE_U).squeeze()
            cube_I = _fits.getdata(CUBE_I).squeeze()
            noise_g71 = _fits.getdata(NOISE_G71).squeeze().astype(np.float32)
            weights_g71 = compute_weights(noise_g71)
        else:
            print("[note] No G71 cubes — using cached qufit posteriors only (no SBI inference).")

        fig, axes_g71 = plt.subplots(1, len(G71_PIXELS), figsize=(7 * len(G71_PIXELS), 7))
        if len(G71_PIXELS) == 1:
            axes_g71 = [axes_g71]

        for idx, (row, col) in enumerate(G71_PIXELS):
            ax_pix = axes_g71[idx]  # placeholder — real code uses subgrid per pixel
            print(f"  Pixel ({row}, {col})")

            # Load qufit cache
            if have_cache[(row, col)]:
                c_data   = np.load(cached[(row, col)])
                rm_qufit = {"RM": c_data["rm"], "amp": c_data["amp"],
                             "chi0_deg": c_data["chi0_deg"]}
                print(f"    QUfit: RM={rm_qufit['RM'].mean():.1f}±{rm_qufit['RM'].std():.1f}")
            else:
                print(f"    [skip] No qufit cache for pixel ({row},{col})")
                continue

            # Run VROOM-SBI if cubes are available
            if have_cubes:
                Q_raw = cube_Q[:, row, col].astype(np.float32)
                U_raw = cube_U[:, row, col].astype(np.float32)
                I_raw = cube_I[:, row, col].astype(np.float32)
                valid = (I_raw > 0) & np.isfinite(I_raw)
                Q_pix = np.where(valid, Q_raw / I_raw, 0.0).astype(np.float32)
                U_pix = np.where(valid, U_raw / I_raw, 0.0).astype(np.float32)
                Q_pix[(weights_g71 == 0) | ~np.isfinite(Q_pix)] = 0.0
                U_pix[(weights_g71 == 0) | ~np.isfinite(U_pix)] = 0.0

                x_g71 = np.concatenate([Q_pix, U_pix])
                if ic_cc == 3:
                    x_g71 = np.concatenate([x_g71, weights_g71])
                x_t_g71 = torch.tensor(x_g71, dtype=torch.float32, device=DEVICE)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    samps_g71 = posterior_cc.sample((N_SBI,), x=x_t_g71).cpu().numpy()
                samps_g71[:, 2] = np.degrees(samps_g71[:, 2])   # chi0 rad→deg
                print(f"    SBI: RM={samps_g71[:, 0].mean():.1f}±{samps_g71[:, 0].std():.1f}")
            else:
                samps_g71 = None

            print(f"    Done (see corner_compare_plots/G71_sobol.pdf when all pixels complete)")

        # Full corner plot (3 panels side by side with subgrid each)
        # See scripts/corner_compare.py::make_corner() for the full implementation
        print("\nFor the full multi-panel corner figure, run:")
        print("  pixi run -e notebooks python scripts/corner_compare.py")

        plt.close("all")


---
## Appendix A: Spectral Shape SBI — Single-Source Demo

Beyond QU-fitting, VROOM-SBI includes a separate Neural Posterior Estimator
for total-intensity **spectral shape** inference.

### Physical model
The spectral SED is parametrised as a log-log cubic polynomial:

$$\log F(\nu) = \alpha \cdot x + \beta \cdot x^2 + \gamma \cdot x^3 + \log F(\nu_0)$$

where $x = \log(\nu/\nu_0)$ and $\nu_0$ is the reference frequency (mid channel).

Parameters:
- **α** — spectral index (power law slope)
- **β** — spectral curvature
- **γ** — spectral curvature derivative

$\log F(\nu_0)$ is not a free parameter — it is computed analytically from the data
by normalising the spectrum at the reference channel.

### Figure
Left: observed spectrum (grey dots) + truth (red) + posterior-predictive draws (blue).  
Right: 3×3 corner plot of the [α, β, γ] posterior.


In [ ]:
# ── Appendix A: spectral shape posterior demo ─────────────────────────────────
# Reproduces: images/spectral_demo.pdf (scripts/plot_spectral_demo.py)

from src.simulator.spectral_simulator import SpectralShapeSimulator
from src.inference.engine import InferenceEngine

FREQ_FILE_SPEC  = REPO / "freq.txt"
MODEL_PATH_SPEC = MODELS_DIR / "spectral_shape_posterior.pt"

if not MODEL_PATH_SPEC.exists():
    print(f"[skip] {MODEL_PATH_SPEC.name} not found")
elif not FREQ_FILE_SPEC.exists():
    print(f"[skip] {FREQ_FILE_SPEC.name} not found")
else:
    # True parameters: alpha=-0.7 (steep spectrum), beta=-0.3 (curved), gamma=0.0
    THETA_SPEC   = np.array([-0.7, -0.3, 0.0], dtype=np.float32)
    NOISE_SPEC   = 0.01      # ~1% rms on fractional flux
    N_SAMP_SPEC  = 2000
    N_PP_SPEC    = 80

    PARAM_LABS_SPEC = [r"$\alpha$", r"$\beta$", r"$\gamma$"]
    N_P_SPEC = 3

    np.random.seed(42); torch.manual_seed(42)

    sim_spec  = SpectralShapeSimulator(str(FREQ_FILE_SPEC))
    freq_ghz  = sim_spec.freq / 1e9
    x_lognu   = sim_spec._log_nu_ratio    # log(nu/nu0) for each channel

    f_true = sim_spec.simulate_noiseless(THETA_SPEC)   # noiseless model
    f_obs  = sim_spec.simulate(THETA_SPEC, noise_sigma=NOISE_SPEC)

    engine_spec = InferenceEngine(device=DEVICE)
    engine_spec.load_spectral_shape_model(MODEL_PATH_SPEC)
    samps_spec, f_nu0 = engine_spec.infer_spectra(
        f_obs,
        weights=np.ones(sim_spec.n_freq, dtype=np.float32),
        n_samples=N_SAMP_SPEC,
    )
    # samps_spec: (N_SAMP_SPEC, 3) — [alpha, beta, gamma]
    print("Truth :", THETA_SPEC)
    print("Median:", np.median(samps_spec, axis=0).round(4))

    idx_pp_spec = np.random.choice(N_SAMP_SPEC, N_PP_SPEC, replace=False)

    # ── Figure: spectrum (left) + 3×3 corner (right) ─────────────────────────
    fig = plt.figure(figsize=(8.0, 3.4))
    ax_spec = fig.add_axes([0.06, 0.15, 0.40, 0.73])

    # 3×3 corner grid
    cl_s, cb_s, cw_s, ch_s = 0.52, 0.10, 0.46, 0.84
    gap_s   = 0.005
    cw_cell = (cw_s - 2 * gap_s) / N_P_SPEC
    ch_cell = (ch_s - 2 * gap_s) / N_P_SPEC
    axes_spec = np.empty((N_P_SPEC, N_P_SPEC), dtype=object)
    for row in range(N_P_SPEC):
        for col in range(N_P_SPEC):
            x0 = cl_s + col * (cw_cell + gap_s)
            y0 = cb_s + (N_P_SPEC - 1 - row) * (ch_cell + gap_s)
            axes_spec[row, col] = fig.add_axes([x0, y0, cw_cell, ch_cell])

    # spectrum panel — posterior predictive draws
    for k in idx_pp_spec:
        a, b, g = samps_spec[k]
        f_pp = np.exp(a * x_lognu + b * x_lognu**2 + g * x_lognu**3) * f_nu0
        ax_spec.plot(freq_ghz, f_pp, color=BLUE, alpha=0.05, lw=0.6, zorder=1)

    ax_spec.plot(freq_ghz, f_obs, ".", ms=1.8, color=GREY, alpha=0.7, label="Observed", zorder=2)
    ax_spec.plot(freq_ghz, f_true, color=RED, lw=1.4, label="Truth", zorder=3)
    ax_spec.set_xlabel(r"$\nu$ (GHz)")
    ax_spec.set_ylabel(r"$F(\nu)$ (normalised)")
    ax_spec.set_title(
        rf"$\alpha={THETA_SPEC[0]:.1f},\ \beta={THETA_SPEC[1]:.1f},\ \gamma={THETA_SPEC[2]:.1f}$",
        pad=4,
    )
    ax_spec.tick_params(direction="in", top=True, right=True)
    ax_spec.grid(True, lw=0.35, alpha=0.4)
    ax_spec.legend(fontsize=7.5, framealpha=0.85, loc="upper right")

    # corner panel
    draw_corner(axes_spec, samps_spec, THETA_SPEC, N_P_SPEC, PARAM_LABS_SPEC)
    axes_spec[0, 0].set_title("Posterior", pad=4, fontsize=9)

    fig.savefig(IMAGES_DIR / "spectral_demo.pdf", bbox_inches="tight", dpi=200)
    plt.show()
    print(f"Saved → {IMAGES_DIR / 'spectral_demo.pdf'}")


---
## Summary of figures

| Figure file | Section | Tier |
|-------------|---------|------|
| `images/sbc_ranks.pdf` | §2.1 SBC | 1 |
| `images/posterior_faraday_thin_n1_recovery.pdf` | §3 | 1 |
| `images/posterior_internal_dispersion_n1_recovery.pdf` | §3 / App A | 1 |
| `images/posterior_external_dispersion_n1_recovery.pdf` | §3 / App A | 1 |
| `images/posterior_burn_slab_n1_recovery.pdf` | §3 / App A | 1 |
| `images/n2_recovery.pdf` | §3.4 N=2 | 1 |
| `images/n2_case2.pdf` | §3.4 N=2 | 1 |
| `images/spectral_index_map.pdf` | §4.3 | 3 |
| `images/rm_maps_2x2.pdf` | §4.4 | 3 |
| `images/comparison_n1_sobol.pdf` | §4.4 | 3 |
| `corner_compare_plots/sim_test_sobol.pdf` | §4.4 |  2 |
| `corner_compare_plots/G71_sobol.pdf` | §4.4 | 2/3 |
| `images/spectral_demo.pdf` | App B | 1 |
